# Run21 Regime Period Documentation

This notebook documents the publication-facing regime buckets used for Run21 evaluation, the exact date boundaries, and the source rationale for each boundary.

## Final Bucket Scheme

| Bucket slug | Start date | End date (exclusive) | Display label | Boundary rationale |
|---|---:|---:|---|---|
| `pre_covid` | start of sample | `2020-03-01` | Pre-COVID (through 2020-02) | Ends before the COVID crash/recession window begins. |
| `covid_crash_recession` | `2020-03-01` | `2020-05-01` | COVID Crash / NBER Recession (2020-03 to 2020-04) | Aligns to the official NBER recession trough in April 2020, with May 2020 as the first month of the new expansion. |
| `covid_recovery` | `2020-05-01` | `2021-01-01` | COVID Recovery (2020-05 to 2020-12) | Captures the rebound period after the NBER trough and before the 2021 post-pandemic rally. |
| `post_pandemic_rally` | `2021-01-01` | `2022-01-01` | Post-Pandemic Rally (2021) | Keeps 2021 as its own market structure regime rather than mixing it with 2020 recovery or 2022 tightening. |
| `inflation_shock_fed_tightening` | `2022-01-01` | `2023-07-27` | Inflation Shock / Fed Tightening (2022 to 2023-07-26) | Covers the high-inflation / tightening cycle through the last Fed hike on July 26, 2023. |
| `disinflation_higher_for_longer` | `2023-07-27` | `2024-09-19` | Disinflation / Higher for Longer (2023-07-27 to 2024-09-18) | Begins after the last hike and ends when the next easing cycle begins. |
| `rate_cut_ai_bull` | `2024-09-19` | open-ended | Rate Cuts / AI Bull (2024-09-19+) | Begins after the first Fed cut on September 18, 2024, effective September 19, 2024. |

## Why The Old Buckets Were Wrong

The earlier notebook treated dates through `2020-08-31` as `covid_crash`. That is too coarse. By then the market had already moved well into recovery, and the S&P 500 had reclaimed its prior high in August 2020.

The practical implication was a logging mismatch: the notebook plan called dates like `2020-06-02` and `2020-08-31` `covid_crash`, while the evaluation engine correctly described them as recovery-period dates. The patched scheme removes that mismatch.

## Source Logic

### 1. NBER recession dating
- NBER identifies the business-cycle peak in **February 2020** and the trough in **April 2020**.
- That makes **May 2020** the clean start of the post-recession recovery bucket.

### 2. 2021 as a separate regime
- 2021 should not be merged into 2020 recovery. It was a distinct post-pandemic rally environment.
- Inflation had already reached **7.0% YoY in December 2021**, which supports treating 2021 as a separate lead-in regime before the full tightening shock.

### 3. Fed tightening cycle boundary
- The first hike in the cycle was **March 16, 2022**.
- The last hike in the cycle was **July 26, 2023**.
- Using `2023-07-27` as the next-day bucket boundary is clean and operational.

### 4. First cut boundary
- The Fed cut rates on **September 18, 2024**.
- Using `2024-09-19` as the next-day bucket boundary cleanly starts the next easing-era bucket.

In [ ]:
REGIME_BUCKET_SPECS = [
    ('pre_covid', '1900-01-01', '2020-03-01', 'Pre-COVID (through 2020-02)'),
    ('covid_crash_recession', '2020-03-01', '2020-05-01', 'COVID Crash / NBER Recession (2020-03 to 2020-04)'),
    ('covid_recovery', '2020-05-01', '2021-01-01', 'COVID Recovery (2020-05 to 2020-12)'),
    ('post_pandemic_rally', '2021-01-01', '2022-01-01', 'Post-Pandemic Rally (2021)'),
    ('inflation_shock_fed_tightening', '2022-01-01', '2023-07-27', 'Inflation Shock / Fed Tightening (2022 to 2023-07-26)'),
    ('disinflation_higher_for_longer', '2023-07-27', '2024-09-19', 'Disinflation / Higher for Longer (2023-07-27 to 2024-09-18)'),
    ('rate_cut_ai_bull', '2024-09-19', '2100-01-01', 'Rate Cuts / AI Bull (2024-09-19+)'),
]

from pandas import Timestamp

def regime_bucket(ts):
    ts = Timestamp(ts)
    for bucket, start_date, end_date, label in REGIME_BUCKET_SPECS:
        if Timestamp(start_date) <= ts < Timestamp(end_date):
            return bucket, label
    return 'unknown', 'Unknown'

for sample in ['2020-02-28', '2020-03-02', '2020-06-02', '2021-05-04', '2022-03-15', '2023-08-01', '2024-10-01']:
    print(sample, '->', regime_bucket(sample))


## Reference List

### Official sources
1. NBER Business Cycle Dating chronology
   - https://www.nber.org/research/business-cycle-dating
2. NBER FAQ on dating procedure and April 2020 trough / May 2020 expansion start
   - https://www.nber.org/research/business-cycle-dating/business-cycle-dating-procedure-frequently-asked-questions
3. Federal Reserve FOMC statement, March 16, 2022
   - https://www.federalreserve.gov/newsevents/pressreleases/monetary20220316a.htm
4. Federal Reserve FOMC statement, July 26, 2023
   - https://www.federalreserve.gov/newsevents/pressreleases/monetary20230726a.htm
5. Federal Reserve FOMC statement, September 18, 2024
   - https://www.federalreserve.gov/newsevents/pressreleases/monetary20240918a.htm
6. BLS CPI, December 2021 (`7.0%` YoY)
   - https://www.bls.gov/news.release/archives/cpi_01122022.htm

### Supporting market-structure reference
7. CNBC on the S&P 500 reclaiming a record high on August 18, 2020
   - https://www.cnbc.com/2020/08/18/sp-500-closes-at-record-high-recovering-its-coronavirus-loss.html

## Writeup Notes

For the paper, describe these as **calendar-defined macro/market regimes** rather than latent or model-inferred regimes. That claim is accurate and defensible.

Recommended wording: 

> We grouped evaluation windows into pre-specified calendar regimes tied to official business-cycle and monetary-policy breakpoints: the COVID recession, post-COVID recovery, 2021 post-pandemic rally, the 2022 to mid-2023 inflation/tightening period, the subsequent disinflation/higher-for-longer phase, and the post-September-2024 rate-cut period.

That is materially stronger than calling them generic `bull` / `bear` regimes without a rule.